<a href="https://colab.research.google.com/github/adhithyalakshman/FLIPKART-GRIDLOCK/blob/main/GRIDLOCK2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [56]:
from google.colab import drive
import pandas as pd
drive.mount('/content/drive')
dfsource=pd.read_csv('/content/drive/MyDrive/FLIPKART_GRIDLOCK/train.csv')
df=dfsource.copy()
df["day"].nunique()



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


2

library  install and  preprocessing


In [58]:
pip install python-geohash scikit-learn lightgbm pandas numpy

In [59]:
!pip install geohash2

In [61]:
import warnings
warnings.filterwarnings("ignore")

import os
import pickle
import numpy as np
import pandas as pd
import geohash2 as gh

from sklearn.cluster     import KMeans
from sklearn.preprocessing import LabelEncoder,StandardScaler
from sklearn.metrics     import mean_squared_error, r2_score
import lightgbm as lgb

configurations for file path


In [63]:
DATA_PATH          = "/content/drive/MyDrive/FLIPKART_GRIDLOCK/train.csv"   # ← change to your path
TEST_CSV_PATH      = None          # ← for testing
N_SPATIAL_CLUSTERS = 50
OOF_FOLDS          = 5
OOF_SMOOTHING      = 20
RANDOM_STATE       = 42
TARGET             = "demand"
# Minimum observations required to trust a geo×slot mean
GEO_SLOT_MIN_COUNT = 3
FEATURE_COLS = [
    # Spatial
    "lat", "lon", "spatial_cluster", "cluster_dist_scaled",   # FIX #6: scaled
    # Geohash encodings
    "geo_l3_enc", "geo_l4_enc", "geo_l5_enc", "geohash_enc", "cluster_enc",
    # Time
    "hour", "minute", "time_slot", "slot_quarter",
    "minutes_since_midnight",
    "hour_sin", "hour_cos", "slot_sin", "slot_cos",
    # Time flags
    "is_peak", "is_morning", "is_night", "is_trough",
    # Road features
    "NumberofLanes", "Temperature",
    "road_highway", "road_street", "is_residential",
    "has_landmark", "trucks_ok",
    "lane_highway", "landmark_highway",
    "RoadType_enc", "Weather_enc",
    # Target-encoded interaction features
    "geo_slot_mean", "cluster_slot_mean", "geo_l4_slot_mean",
    "geo_demand_std", "geo_peak_mean", "neighbor_mean",
    # Lag features
    "demand_lag_1", "demand_lag_2", "demand_lag_3",
    "demand_lag_4", "demand_lag_6", "demand_lag_8",
    "demand_lag1_sq",
    # Rolling features
    "roll_mean_4",  "roll_mean_8",  "roll_mean_16",
    "roll_std_4",   "roll_std_8",   "roll_std_16",   # FIX #1 #8: 0-filled
    "roll_min_8",   "roll_max_8",
    # Trend
    "demand_trend_4",                                # FIX #3: fast slope
]


 LOAD & PARSE DATA (columns only — no feature engineering )


In [64]:
def load_and_parse(path: str) -> pd.DataFrame:
    print(f"\n[1] Loading '{path}' …")
    df = pd.read_csv(path)
    print(f"    Raw shape: {df.shape}")

    df["hour"]      = df["timestamp"].apply(lambda x: int(str(x).split(":")[0]))
    df["minute"]    = df["timestamp"].apply(lambda x: int(str(x).split(":")[1]))
    df["time_slot"] = df["hour"] * 4 + df["minute"] // 15   # 0–95

    df["geo_l3"] = df["geohash"].str[:3]   # NEW: coarser fallback zone
    df["geo_l4"] = df["geohash"].str[:4]
    df["geo_l5"] = df["geohash"].str[:5]

    return df

TRAIN / TEST SPLIT

In [65]:

def split_train_test(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Split BEFORE any demand-based feature computation.
    Day 48 → train (~90%).  Day 49 → test (~10%).
    Sort by [geohash, time_slot] so lag shifts are chronologically correct
    within each geohash series.
    """
    train = df[df["day"] == 48].copy()
    test  = df[df["day"] == 49].copy()

    train = train.sort_values(["geohash", "time_slot"]).reset_index(drop=True)
    test  = test.sort_values( ["geohash", "time_slot"]).reset_index(drop=True)

    ratio = len(test) / len(df) * 100
    print(f"\n[2] Train/test split")
    print(f"    Train (day 48): {len(train):>6,} rows  ({100-ratio:.1f}%)")
    print(f"    Test  (day 49): {len(test):>6,} rows  ({ratio:.1f}%)")

    unseen = set(test["geohash"].unique()) - set(train["geohash"].unique())
    if unseen:
        print(f"    ⚠  {len(unseen)} test geohashes not seen in train")

    return train, test

STEP 3 — GEOHASH DECODE  →  lat / lon

In [81]:
def fit_geohash_coords(train: pd.DataFrame) -> dict:
    """Decode all geohashes seen in training data."""
    print("\n[3] Decoding geohashes to lat/lon …")
    coords_map = {g: gh.decode(g) for g in train["geohash"].unique()}
    print(f"    Decoded {len(coords_map):,} unique geohashes")
    return coords_map


def apply_geohash_coords(df: pd.DataFrame, coords_map: dict) -> pd.DataFrame:
    """
    Apply coords map to any dataframe.
    Geohashes not seen in training are decoded on-the-fly (still leak-free
    since decode() uses only the geohash string, not demand values).
    """
    def get_lat(g):
        return coords_map[g][0] if g in coords_map else gh.decode(g)[0]
    def get_lon(g):
        return coords_map[g][1] if g in coords_map else gh.decode(g)[1]

    df["lat"] = df["geohash"].apply(get_lat).astype(float)
    df["lon"] = df["geohash"].apply(get_lon).astype(float)
    return df

STEP 4  SPATIAL CLUSTERING  (fitted on train only)

In [67]:
def fit_spatial_clusters(train: pd.DataFrame) -> tuple[KMeans, dict, StandardScaler]:
    """
    FIX #6: cluster_dist is in degree-units (roughly 0.001–0.5) but the
    scale can dominate tree splits compared to 0/1 flag features.
    We fit a StandardScaler on train distances and return it so test
    distances can be normalised with the same parameters.
    """
    print(f"\n[4] Fitting spatial KMeans (k={N_SPATIAL_CLUSTERS}) on train …")
    unique_coords = train[["geohash", "lat", "lon"]].drop_duplicates("geohash")
    km = KMeans(n_clusters=N_SPATIAL_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
    km.fit(unique_coords[["lat", "lon"]])
    cluster_map = dict(zip(unique_coords["geohash"], km.labels_))

    # Fit distance scaler on train unique geohash distances
    raw_dists = km.transform(unique_coords[["lat", "lon"]]).min(axis=1).reshape(-1, 1)
    dist_scaler = StandardScaler()
    dist_scaler.fit(raw_dists)

    print(f"    {N_SPATIAL_CLUSTERS} spatial clusters created")
    return km, cluster_map, dist_scaler


def apply_spatial_clusters(
    df: pd.DataFrame,
    km: KMeans,
    cluster_map: dict,
    dist_scaler: StandardScaler,
) -> pd.DataFrame:
    def get_cluster(row):
        if row["geohash"] in cluster_map:
            return cluster_map[row["geohash"]]
        return int(km.predict([[row["lat"], row["lon"]]])[0])

    df["spatial_cluster"] = df.apply(get_cluster, axis=1)

    # FIX #6: compute raw distance then scale using train-fitted scaler
    raw_dist = km.transform(df[["lat", "lon"]]).min(axis=1).reshape(-1, 1)
    df["cluster_dist_scaled"] = dist_scaler.transform(raw_dist).ravel()
    return df

STEP 5 — MISSING VALUE IMPUTATION  (maps fitted on train only)

In [68]:
def fit_imputation_maps(train: pd.DataFrame) -> dict:
    print("\n[5] Fitting imputation maps on train …")
    m = {}

    # Temperature cascade: geohash → geo_l3 → geo_l4 → global
    m["temp_by_geohash"] = train.groupby("geohash")["Temperature"].median().to_dict()
    m["temp_by_geo_l3"]  = train.groupby("geo_l3")["Temperature"].median().to_dict()
    m["temp_by_geo_l4"]  = train.groupby("geo_l4")["Temperature"].median().to_dict()
    m["temp_global"]     = float(train["Temperature"].median())

    m["road_by_geohash"] = (
        train.dropna(subset=["RoadType"])
        .groupby("geohash")["RoadType"]
        .agg(lambda x: x.mode()[0])
        .to_dict()
    )
    m["road_global"] = train["RoadType"].mode()[0]

    m["weather_by_geohash"] = (
        train.dropna(subset=["Weather"])
        .groupby("geohash")["Weather"]
        .agg(lambda x: x.mode()[0])
        .to_dict()
    )
    m["weather_global"] = train["Weather"].mode()[0]

    nulls = train[["Temperature", "RoadType", "Weather"]].isnull().sum()
    print(f"    Train nulls before imputation: {nulls.to_dict()}")
    return m


def apply_imputation(df: pd.DataFrame, m: dict) -> pd.DataFrame:
    # Temperature: 4-level cascade
    mask = df["Temperature"].isna()
    df.loc[mask, "Temperature"] = df.loc[mask, "geohash"].map(m["temp_by_geohash"])
    mask = df["Temperature"].isna()
    df.loc[mask, "Temperature"] = df.loc[mask, "geo_l3"].map(m["temp_by_geo_l3"])
    mask = df["Temperature"].isna()
    df.loc[mask, "Temperature"] = df.loc[mask, "geo_l4"].map(m["temp_by_geo_l4"])
    df["Temperature"] = df["Temperature"].fillna(m["temp_global"])

    mask = df["RoadType"].isna()
    df.loc[mask, "RoadType"] = df.loc[mask, "geohash"].map(m["road_by_geohash"])
    df["RoadType"] = df["RoadType"].fillna(m["road_global"])

    mask = df["Weather"].isna()
    df.loc[mask, "Weather"] = df.loc[mask, "geohash"].map(m["weather_by_geohash"])
    df["Weather"] = df["Weather"].fillna(m["weather_global"])

    nulls = df[["Temperature", "RoadType", "Weather"]].isnull().sum()
    print(f"    Nulls after imputation: {nulls.to_dict()}")
    return df



STEP 6 — LAG & ROLLING FEATURES

In [69]:
def _fast_slope(series: pd.Series, window: int) -> pd.Series:
    """
    FIX #3: Replace np.polyfit inside rolling (O(n·w·log w)) with a
    closed-form OLS slope using only sum(x), sum(x²), sum(y), sum(xy).

    For integer x = [0, 1, …, w-1]:
        slope = (w·Σxy - Σx·Σy) / (w·Σx² - (Σx)²)

    This is fully vectorised via pandas rolling — no apply() overhead.
    Produces the same result as polyfit degree-1 but ~20–50× faster.
    """
    w = window
    # Precompute constant denominators for this window size
    sum_x  = w * (w - 1) / 2          # sum of [0,1,...,w-1]
    sum_x2 = w * (w - 1) * (2*w - 1) / 6
    denom  = w * sum_x2 - sum_x ** 2  # always > 0 for w >= 2

    # Rolling sums of y and x*y (x is the position index 0..w-1)
    # We approximate Σ(i * y_i) using the rolling weighted sum trick:
    # weight for the oldest value in window = 0, newest = w-1
    weights = np.arange(w, dtype=float)

    roll_sum_y  = series.rolling(w, min_periods=w).sum()
    roll_sum_xy = series.rolling(w, min_periods=w).apply(
        lambda arr: float(np.dot(weights, arr)), raw=True
    )

    slope = (w * roll_sum_xy - sum_x * roll_sum_y) / denom
    return slope


def build_lag_features_train(train: pd.DataFrame) -> pd.DataFrame:
    """
    FIX #1: roll_std_* NaN (from ddof=1 on 1-element windows) must be
    filled with 0.0 — NOT with geo_mean.  std=0 correctly signals
    'no variability observed' which is semantically accurate.

    FIX #8: Changed min_periods from 1 → 2 for std columns so that
    single-element windows don't even enter the std computation and
    instead resolve to NaN which we then fill with 0.0.

    FIX #3: demand_trend_4 uses _fast_slope instead of polyfit.
    """
    print("\n[6] Building lag features for train …")
    g = train.groupby("geohash")["demand"]

    # ── Point-in-time lags ──────────────────────────────────────────────────
    for lag in [1, 2, 3, 4, 6, 8]:
        train[f"demand_lag_{lag}"] = g.shift(lag)

    # ── Rolling mean (min_periods=1 is fine — mean of 1 value is the value) ─
    for w in [4, 8, 16]:
        train[f"roll_mean_{w}"] = g.transform(
            lambda x: x.shift(1).rolling(w, min_periods=1).mean()
        )

    # ── Rolling std (FIX #1 #8: min_periods=2; NaN → 0.0) ──────────────────
    for w in [4, 8, 16]:
        train[f"roll_std_{w}"] = g.transform(
            lambda x: x.shift(1).rolling(w, min_periods=2).std(ddof=1)
        )
        # FIX #1: fill with 0.0, not geo_mean.  std=0 for sparse history is
        # the correct neutral value — it does NOT proxy for demand level.
        train[f"roll_std_{w}"] = train[f"roll_std_{w}"].fillna(0.0)

    # ── Rolling min / max ────────────────────────────────────────────────────
    train["roll_min_8"] = g.transform(
        lambda x: x.shift(1).rolling(8, min_periods=1).min()
    )
    train["roll_max_8"] = g.transform(
        lambda x: x.shift(1).rolling(8, min_periods=1).max()
    )

    # ── Demand trend: fast vectorised slope over last 4 slots (FIX #3) ──────
    def fast_slope_transform(x: pd.Series) -> pd.Series:
        shifted = x.shift(1)
        slope   = _fast_slope(shifted, window=4)
        return slope

    train["demand_trend_4"] = g.transform(fast_slope_transform)

    # ── Nonlinear feature ────────────────────────────────────────────────────
    train["demand_lag1_sq"] = train["demand_lag_1"] ** 2

    # ── Fill remaining NaNs ──────────────────────────────────────────────────
    # For lags: first few rows per geohash have no history → fill with geo mean
    # (still a demand-based fill, but only for lag/mean columns — NOT for std)
    geo_mean = train.groupby("geohash")[TARGET].transform("mean")

    lag_cols  = [f"demand_lag_{l}" for l in [1, 2, 3, 4, 6, 8]]
    mean_cols = [f"roll_mean_{w}"  for w in [4, 8, 16]]
    minmax_cols = ["roll_min_8", "roll_max_8"]
    other_cols  = ["demand_trend_4", "demand_lag1_sq"]

    for col in lag_cols + mean_cols + minmax_cols:
        train[col] = train[col].fillna(geo_mean)

    # Trend slope: NaN rows (first 4 per geohash) → fill with 0 (no trend known)
    for col in other_cols:
        train[col] = train[col].fillna(0.0)

    # Verify — std columns should now all be 0.0, never geo_mean
    remaining = train[FEATURE_COLS if False else
                      [c for c in train.columns if c.startswith("roll_std")]
                     ].isnull().sum().sum()
    print(f"    Remaining NaNs in roll_std columns: {remaining}  (expected 0)")
    total_nan = train[[f"demand_lag_{l}" for l in [1,2,3,4,6,8]] +
                       [f"roll_mean_{w}" for w in [4,8,16]] +
                       [f"roll_std_{w}"  for w in [4,8,16]] +
                       ["roll_min_8","roll_max_8","demand_trend_4","demand_lag1_sq"]
                      ].isnull().sum().sum()
    print(f"    Total NaNs in all lag/rolling columns: {total_nan}  (expected 0)")
    return train


def build_lag_features_test(test: pd.DataFrame, train: pd.DataFrame) -> pd.DataFrame:
    """
    FIX #1 #8: hist_roll_std returns 0.0 for windows with < 2 values.
    FIX #3: trend slope uses the same fast closed-form formula.
    History buffer extended to 24 values to support lag_6.
    """
    print("\n[6-TEST] Building lag features for test (using train history) …")

    history_buffer: dict[str, np.ndarray] = (
        train.sort_values(["geohash", "time_slot"])
        .groupby("geohash")["demand"]
        .apply(lambda x: x.values[-24:])   # extended from 16 → 24
        .to_dict()
    )

    global_mean  = float(train[TARGET].mean())
    geo_mean_map = train.groupby("geohash")[TARGET].mean().to_dict()

    def _fallback(geohash: str) -> float:
        return geo_mean_map.get(geohash, global_mean)

    def hist_lag(geohash: str, lag: int) -> float:
        hist = history_buffer.get(geohash, np.array([]))
        if len(hist) >= lag:
            return float(hist[-lag])
        return _fallback(geohash)

    def hist_roll_mean(geohash: str, w: int) -> float:
        hist = history_buffer.get(geohash, np.array([]))
        if len(hist) == 0:
            return _fallback(geohash)
        return float(np.mean(hist[-w:]))

    def hist_roll_std(geohash: str, w: int) -> float:
        """FIX #1 #8: return 0.0 when fewer than 2 observations available."""
        hist = history_buffer.get(geohash, np.array([]))
        window = hist[-w:]
        if len(window) < 2:
            return 0.0   # ← was returning geo_mean proxy — now correctly 0
        return float(np.std(window, ddof=1))

    def hist_roll_min(geohash: str, w: int) -> float:
        hist = history_buffer.get(geohash, np.array([]))
        if len(hist) == 0:
            return _fallback(geohash)
        return float(np.min(hist[-w:]))

    def hist_roll_max(geohash: str, w: int) -> float:
        hist = history_buffer.get(geohash, np.array([]))
        if len(hist) == 0:
            return _fallback(geohash)
        return float(np.max(hist[-w:]))

    def hist_slope(geohash: str) -> float:
        """FIX #3: fast closed-form slope, matching _fast_slope in train."""
        hist = history_buffer.get(geohash, np.array([]))
        window = hist[-4:] if len(hist) >= 4 else None
        if window is None or len(window) < 2:
            return 0.0
        w = len(window)
        x = np.arange(w, dtype=float)
        # Closed-form OLS slope
        mx = x.mean()
        my = window.mean()
        slope = np.sum((x - mx) * (window - my)) / np.sum((x - mx) ** 2)
        return float(slope)

    for lag in [1, 2, 3, 4, 6, 8]:
        test[f"demand_lag_{lag}"] = test["geohash"].apply(
            lambda g, l=lag: hist_lag(g, l))

    for w in [4, 8, 16]:
        test[f"roll_mean_{w}"] = test["geohash"].apply(
            lambda g, w=w: hist_roll_mean(g, w))
        test[f"roll_std_{w}"]  = test["geohash"].apply(
            lambda g, w=w: hist_roll_std(g, w))   # FIX #1

    test["roll_min_8"]     = test["geohash"].apply(lambda g: hist_roll_min(g, 8))
    test["roll_max_8"]     = test["geohash"].apply(lambda g: hist_roll_max(g, 8))
    test["demand_trend_4"] = test["geohash"].apply(hist_slope)   # FIX #3
    test["demand_lag1_sq"] = test["demand_lag_1"] ** 2

    print(f"    Lag features built from Day 48 history buffer (24 slots)")
    return test

STEP 7 — CYCLIC TIME ENCODING

In [70]:
def add_cyclic_and_time_flags(df: pd.DataFrame) -> pd.DataFrame:
    df["hour_sin"]  = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"]  = np.cos(2 * np.pi * df["hour"] / 24)
    df["slot_sin"]  = np.sin(2 * np.pi * df["time_slot"] / 96)
    df["slot_cos"]  = np.cos(2 * np.pi * df["time_slot"] / 96)

    df["is_peak"]    = df["hour"].between(9, 13).astype(int)
    df["is_morning"] = df["hour"].between(4, 8).astype(int)
    df["is_night"]   = ((df["hour"] >= 21) | (df["hour"] <= 3)).astype(int)
    df["is_trough"]  = df["hour"].between(17, 20).astype(int)

    df["slot_quarter"]          = (df["time_slot"] // 24).astype(int)   # 0–3
    df["minutes_since_midnight"] = df["hour"] * 60 + df["minute"]
    return df

STEP 8 — DOMAIN FLAG FEATURES

In [71]:
def add_domain_flags(df: pd.DataFrame) -> pd.DataFrame:
    df["road_highway"]    = (df["RoadType"] == "Highway").astype(int)
    df["road_street"]     = (df["RoadType"] == "Street").astype(int)
    df["is_residential"]  = (
        (df["road_highway"] == 0) & (df["road_street"] == 0)
    ).astype(int)
    df["has_landmark"]    = (df["Landmarks"] == "Yes").astype(int)
    df["trucks_ok"]       = (df["LargeVehicles"] == "Allowed").astype(int)

    # Crossed features (must come after the base flags above)
    df["lane_highway"]    = df["NumberofLanes"] * df["road_highway"]
    df["landmark_highway"]= df["has_landmark"]  * df["road_highway"]
    return df

 STEP 9 — LOCATION × TIME INTERACTION FEATURES  (fitted on train only)

In [72]:
def fit_interaction_maps(train: pd.DataFrame) -> dict:
    """
    FIX #4: geo_slot_mean is count-gated.  Cells with fewer than
    GEO_SLOT_MIN_COUNT observations get the geohash-level mean instead.
    This prevents the model from memorising demand for sparse (geohash, slot)
    combinations that happen to appear in both train and test by coincidence.

    FIX #2: neighbor_mean now uses a weighted average (weight = 1/distance
    to centroid) and falls back to geo_l4_mean when fewer than 2 valid
    neighbours are found in the training geo_mean map.
    """
    print("\n[9] Fitting location × time interaction maps on train …")
    m = {}

    # ── Existing maps ────────────────────────────────────────────────────────
    geo_slot_stats = train.groupby(["geohash", "time_slot"])[TARGET].agg(
        ["mean", "count"]
    )
    # FIX #4: replace low-count cells with NaN so fallback chain is used
    geo_slot_stats.loc[geo_slot_stats["count"] < GEO_SLOT_MIN_COUNT, "mean"] = np.nan
    m["geo_slot_mean"] = geo_slot_stats["mean"].dropna().to_dict()

    m["cluster_slot_mean"] = (
        train.groupby(["spatial_cluster", "time_slot"])[TARGET].mean().to_dict()
    )
    m["geo_mean"]     = train.groupby("geohash")[TARGET].mean().to_dict()
    m["geo_l4_mean"]  = train.groupby("geo_l4")[TARGET].mean().to_dict()
    m["cluster_mean"] = train.groupby("spatial_cluster")[TARGET].mean().to_dict()
    m["global_mean"]  = float(train[TARGET].mean())

    # ── New: geo_l4 × slot mean ──────────────────────────────────────────────
    m["geo_l4_slot_mean"] = (
        train.groupby(["geo_l4", "time_slot"])[TARGET].mean().to_dict()
    )

    # ── New: geohash demand std ──────────────────────────────────────────────
    m["geo_demand_std"] = (
        train.groupby("geohash")[TARGET].std().fillna(0.0).to_dict()
    )

    # ── New: geohash × is_peak mean ─────────────────────────────────────────
    # Requires is_peak to already be computed — call add_cyclic_and_time_flags
    # BEFORE fit_interaction_maps in main().
    if "is_peak" in train.columns:
        m["geo_peak_mean"] = (
            train.groupby(["geohash", "is_peak"])[TARGET].mean().to_dict()
        )
    else:
        m["geo_peak_mean"] = {}

    # ── New: neighbor demand map (FIX #2) ────────────────────────────────────
    # Use count-weighted fallback: only include neighbours with >= 5 train rows.
    # Fall back to geo_l4_mean when < 2 valid neighbours.
    geo_mean     = m["geo_mean"]
    geo_l4_mean  = m["geo_l4_mean"]
    global_mean  = m["global_mean"]

    geo_counts   = train.groupby("geohash")[TARGET].count().to_dict()
    MIN_NEIGHBOR_OBS = 5   # ignore neighbours with < 5 observations

    neighbor_map: dict[str, float] = {}
    for g in train["geohash"].unique():
        try:
            raw_neighbors = [n for n in gh.expand(g) if n != g]
        except Exception:
            neighbor_map[g] = geo_mean.get(g, global_mean)
            continue

        # FIX #2: filter to neighbours present in train with enough observations
        valid = [
            n for n in raw_neighbors
            if n in geo_mean and geo_counts.get(n, 0) >= MIN_NEIGHBOR_OBS
        ]

        if len(valid) < 2:
            # Sparse zone — fall back to coarser geo_l4 mean
            geo_l4_key = g[:4]
            neighbor_map[g] = geo_l4_mean.get(geo_l4_key, global_mean)
        else:
            # Simple mean over valid neighbours (count-weighted optional but
            # simple mean is more robust against outlier neighbours here)
            neighbor_map[g] = float(np.mean([geo_mean[n] for n in valid]))

    m["neighbor_mean"] = neighbor_map
    print(
        f"    geo_slot_mean entries (count-gated ≥{GEO_SLOT_MIN_COUNT}): "
        f"{len(m['geo_slot_mean']):,}"
    )
    return m


def apply_interaction_maps(df: pd.DataFrame, m: dict) -> pd.DataFrame:
    global_mean = m["global_mean"]

    # geo_slot_mean — fallback: geohash mean → global mean
    df["geo_slot_mean"] = df.apply(
        lambda r: m["geo_slot_mean"].get(
            (r["geohash"], r["time_slot"]),
            m["geo_mean"].get(r["geohash"], global_mean)
        ),
        axis=1,
    )

    # cluster_slot_mean
    df["cluster_slot_mean"] = df.apply(
        lambda r: m["cluster_slot_mean"].get(
            (r["spatial_cluster"], r["time_slot"]),
            m["cluster_mean"].get(r["spatial_cluster"], global_mean)
        ),
        axis=1,
    )

    # geo_l4_slot_mean — fallback: geo_l4 mean → global mean
    df["geo_l4_slot_mean"] = df.apply(
        lambda r: m["geo_l4_slot_mean"].get(
            (r["geo_l4"], r["time_slot"]),
            m["geo_l4_mean"].get(r["geo_l4"], global_mean)
        ),
        axis=1,
    )

    # geo_demand_std — 0 for unseen geohashes
    df["geo_demand_std"] = df["geohash"].map(m["geo_demand_std"]).fillna(0.0)

    # geo_peak_mean — fallback: geo_mean → global mean
    if m["geo_peak_mean"]:
        df["geo_peak_mean"] = df.apply(
            lambda r: m["geo_peak_mean"].get(
                (r["geohash"], r["is_peak"]),
                m["geo_mean"].get(r["geohash"], global_mean)
            ),
            axis=1,
        )
    else:
        df["geo_peak_mean"] = df["geohash"].map(m["geo_mean"]).fillna(global_mean)

    # neighbor_mean — fallback: global mean for completely unseen geohashes
    df["neighbor_mean"] = df["geohash"].map(m["neighbor_mean"]).fillna(global_mean)

    return df

STEP 10 — CATEGORICAL ENCODING  (OOF on train, map-based on test)

In [73]:
def time_aware_oof_target_encode(
    train:     pd.DataFrame,
    col:       str,
    n_folds:   int = OOF_FOLDS,
    smoothing: int = OOF_SMOOTHING,
) -> pd.Series:
    """
    FIX #7: Verified — this function does NOT use random KFold internally.
    It splits the sorted unique time_slots into n_folds consecutive blocks
    and uses strictly past slots as the training set for each fold.

    Assertions added to catch any accidental future leakage during
    development (e.g. if someone swaps this for sklearn KFold).
    """
    slots      = sorted(train["time_slot"].unique())
    fold_size  = len(slots) // n_folds
    global_mean = float(train[TARGET].mean())

    encoded = pd.Series(np.nan, index=train.index, dtype=float)

    for fold in range(n_folds):
        val_start = fold * fold_size
        val_end   = (fold + 1) * fold_size if fold < n_folds - 1 else len(slots)
        val_slots = set(slots[val_start:val_end])

        # FIX #7: strict past-only guarantee — assert no future leakage
        tr_mask  = train["time_slot"] < min(val_slots)
        val_mask = train["time_slot"].isin(val_slots)

        # Sanity check: val rows must have strictly higher slots than all train rows
        if tr_mask.sum() > 0:
            assert train.loc[tr_mask,  "time_slot"].max() < min(val_slots), \
                f"OOF fold {fold}: training slots overlap with validation slots!"
            assert train.loc[val_mask, "time_slot"].min() > train.loc[tr_mask, "time_slot"].max(), \
                f"OOF fold {fold}: validation slots precede training slots!"

        if tr_mask.sum() == 0:
            encoded[val_mask] = global_mean
            continue

        fold_train = train[tr_mask]
        stats = fold_train.groupby(col)[TARGET].agg(["mean", "count"])
        smooth = (
            (stats["mean"] * stats["count"] + global_mean * smoothing) /
            (stats["count"] + smoothing)
        )
        encoded[val_mask] = train.loc[val_mask, col].map(smooth).fillna(global_mean)

    return encoded.fillna(global_mean)


def fit_target_encoding_maps(train: pd.DataFrame) -> dict:
    te_maps = {}
    global_mean = float(train[TARGET].mean())
    te_maps["global_mean"] = global_mean

    for col in ["geohash", "spatial_cluster", "geo_l3", "geo_l4", "geo_l5"]:
        stats = train.groupby(col)[TARGET].agg(["mean", "count"])
        smooth = (
            (stats["mean"] * stats["count"] + global_mean * OOF_SMOOTHING) /
            (stats["count"] + OOF_SMOOTHING)
        )
        te_maps[f"{col}_enc_map"] = smooth.to_dict()

    return te_maps


def fit_label_encoders(train: pd.DataFrame) -> dict:
    les = {}
    for col in ["RoadType", "Weather"]:
        le = LabelEncoder()
        le.fit(train[col].astype(str))
        les[col] = le
    return les


def apply_encoding_train(
    train: pd.DataFrame,
) -> tuple[pd.DataFrame, dict, dict]:
    print("\n[10] Encoding categorical features for train (time-aware OOF) …")

    for col in ["geohash", "geo_l3", "geo_l4", "geo_l5"]:
        train[f"{col}_enc"] = time_aware_oof_target_encode(train, col)

    train["cluster_enc"] = time_aware_oof_target_encode(train, "spatial_cluster")

    te_maps = fit_target_encoding_maps(train)
    les     = fit_label_encoders(train)

    for col, le in les.items():
        train[f"{col}_enc"] = le.transform(train[col].astype(str))

    print("    Train encoding complete")
    return train, te_maps, les


def apply_encoding_test(
    test:    pd.DataFrame,
    te_maps: dict,
    les:     dict,
) -> pd.DataFrame:
    print("\n[10-TEST] Encoding categorical features for test …")
    global_mean = te_maps["global_mean"]

    for col in ["geohash", "geo_l3", "geo_l4", "geo_l5"]:
        test[f"{col}_enc"] = test[col].map(
            te_maps[f"{col}_enc_map"]
        ).fillna(global_mean)

    test["cluster_enc"] = test["spatial_cluster"].map(
        te_maps["spatial_cluster_enc_map"]
    ).fillna(global_mean)

    for col, le in les.items():
        known = set(le.classes_)
        test[f"{col}_enc"] = test[col].apply(
            lambda x: int(le.transform([str(x)])[0]) if str(x) in known else -1
        )

    return test

STEP 11 — FEATURE MATRIX ASSEMBLY

In [74]:
def get_feature_matrix(
    df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.Series | None]:
    available = [c for c in FEATURE_COLS if c in df.columns]
    missing   = [c for c in FEATURE_COLS if c not in df.columns]
    if missing:
        print(f"    ⚠  Missing feature columns (skipped): {missing}")

    X = df[available].copy()
    y = df[TARGET].copy() if TARGET in df.columns else None

    nan_count = X.isnull().sum().sum()
    if nan_count > 0:
        print(f"    ⚠  {nan_count} NaN cells — filling with column median")
        X = X.fillna(X.median(numeric_only=True))

    return X, y


STEP 12 — MODEL TRAINING

In [75]:
def train_lightgbm(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_val:   pd.DataFrame,
    y_val:   pd.Series,
    extra_params: dict | None = None,
) -> lgb.Booster:
    """
    FIX #4 (partial): higher lambda_l1/l2 (0.5 each) adds regularisation
    that partially offsets the memorisation risk from strong target-encoded
    features like geo_slot_mean.

    FIX #5: Optuna search space is capped (num_leaves ≤ 255, max_depth ≤ 10)
    to avoid over-parameterising a model that already has strong memorisation
    features.
    """
    print("\n[12] Training LightGBM …")

    params = {
        "objective":         "regression",
        "metric":            "rmse",
        "num_leaves":        127,          # safe default; tune up to 255 max
        "max_depth":         10,           # FIX #5: cap at 10
        "learning_rate":     0.03,
        "feature_fraction":  0.8,
        "bagging_fraction":  0.8,
        "bagging_freq":      5,
        "min_child_samples": 30,
        "lambda_l1":         0.5,          # FIX #4: tightened from 0.1
        "lambda_l2":         0.5,          # FIX #4: tightened from 0.1
        "min_gain_to_split": 0.01,         # prevents trivial splits
        "verbose":           -1,
        "seed":              RANDOM_STATE,
    }

    if extra_params:
        params.update(extra_params)

    dtrain = lgb.Dataset(X_train, label=y_train)
    dval   = lgb.Dataset(X_val,   label=y_val, reference=dtrain)

    model = lgb.train(
        params,
        dtrain,
        num_boost_round=3000,
        valid_sets=[dval],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=200),
        ],
    )
    print(f"    Best iteration: {model.best_iteration}")
    return model


def tune_lightgbm(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_val:   pd.DataFrame,
    y_val:   pd.Series,
    n_trials: int = 80,
) -> dict:
    """
    FIX #5: Search space is deliberately conservative.
    - num_leaves capped at 255 (not 511)
    - max_depth capped at 10 (not 14)
    - min_child_samples lower bound at 20 (prevents micro-leaves)
    - lambda bounds start at 0.1 to ensure regularisation is always present
    """
    try:
        import optuna
        optuna.logging.set_verbosity(optuna.logging.WARNING)
    except ImportError:
        print("    optuna not installed — run: pip install optuna")
        return {}

    def objective(trial):
        params = {
            "objective":         "regression",
            "metric":            "rmse",
            "verbosity":         -1,
            "seed":              RANDOM_STATE,
            "num_leaves":        trial.suggest_int("num_leaves", 63, 255),    # FIX #5
            "max_depth":         trial.suggest_int("max_depth", 6, 10),       # FIX #5
            "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
            "feature_fraction":  trial.suggest_float("feature_fraction", 0.5, 1.0),
            "bagging_fraction":  trial.suggest_float("bagging_fraction", 0.5, 1.0),
            "bagging_freq":      trial.suggest_int("bagging_freq", 1, 10),
            "min_child_samples": trial.suggest_int("min_child_samples", 20, 80),   # FIX #5
            "lambda_l1":         trial.suggest_float("lambda_l1", 0.1, 5.0, log=True),
            "lambda_l2":         trial.suggest_float("lambda_l2", 0.1, 5.0, log=True),
            "min_gain_to_split": trial.suggest_float("min_gain_to_split", 0.0, 0.5),
        }
        dtrain = lgb.Dataset(X_train, label=y_train)
        dval   = lgb.Dataset(X_val,   label=y_val, reference=dtrain)
        model  = lgb.train(
            params, dtrain,
            num_boost_round=2000,
            valid_sets=[dval],
            callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)],
        )
        preds = np.clip(model.predict(X_val), 0.0, 1.0)
        return float(np.sqrt(mean_squared_error(y_val, preds)))

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    print(f"\n    Best RMSE: {study.best_value:.5f}")
    print(f"    Best params: {study.best_params}")
    return study.best_params

STEP 13 — EVALUATION & FEATURE IMPORTANC

In [76]:
def evaluate(
    model: lgb.Booster,
    X_val: pd.DataFrame,
    y_val: pd.Series,
    label: str = "Validation",
    df_val: pd.DataFrame | None = None,
) -> dict:
    preds = model.predict(X_val, num_iteration=model.best_iteration)
    preds = np.clip(preds, 0.0, 1.0)

    rmse = float(np.sqrt(mean_squared_error(y_val, preds)))
    r2   = float(r2_score(y_val, preds))
    mae  = float(np.mean(np.abs(y_val - preds)))

    print(f"\n{'='*50}")
    print(f"  {label}")
    print(f"{'='*50}")
    print(f"  R²   : {r2:.5f}")
    print(f"  RMSE : {rmse:.5f}")
    print(f"  MAE  : {mae:.5f}")
    print(f"{'='*50}")

    # Per-slot RMSE breakdown — surfaces where the model is weakest
    if df_val is not None and "time_slot" in df_val.columns:
        slot_rmse = (
            pd.DataFrame({"time_slot": df_val["time_slot"].values,
                          "resid_sq":  (y_val.values - preds) ** 2})
            .groupby("time_slot")["resid_sq"]
            .mean()
            .apply(np.sqrt)
            .sort_values(ascending=False)
        )
        print(f"\n  Worst 10 time slots by RMSE:")
        print(slot_rmse.head(10).to_string())

    return {"r2": r2, "rmse": rmse, "mae": mae}


def print_feature_importance(
    model: lgb.Booster,
    X: pd.DataFrame,
    top_n: int = 20,
) -> pd.DataFrame:
    imp = pd.DataFrame({
        "feature":    X.columns,
        "importance": model.feature_importance(importance_type="gain"),
    }).sort_values("importance", ascending=False)
    print(f"\nTop {top_n} features by gain:")
    print(imp.head(top_n).to_string(index=False))
    return imp

STEP 14 — FULL PREPROCESSING PIPELINE FOR EXTERNAL TEST FILE

In [77]:
def preprocess_external_test(
    test_path: str,
    train_df:  pd.DataFrame,
    artifacts: dict,
) -> pd.DataFrame:
    print(f"\n{'='*55}")
    print(f"  Preprocessing external test file: {test_path}")
    print(f"{'='*55}")

    df = pd.read_csv(test_path)

    # Step 1
    df["hour"]      = df["timestamp"].apply(lambda x: int(str(x).split(":")[0]))
    df["minute"]    = df["timestamp"].apply(lambda x: int(str(x).split(":")[1]))
    df["time_slot"] = df["hour"] * 4 + df["minute"] // 15
    df["geo_l3"]    = df["geohash"].str[:3]
    df["geo_l4"]    = df["geohash"].str[:4]
    df["geo_l5"]    = df["geohash"].str[:5]

    df = apply_geohash_coords(df, artifacts["coords_map"])
    df = apply_spatial_clusters(
        df, artifacts["km"], artifacts["cluster_map"], artifacts["dist_scaler"]
    )
    df = apply_imputation(df, artifacts["imputation_maps"])
    df = build_lag_features_test(df, train_df)
    df = add_cyclic_and_time_flags(df)
    df = add_domain_flags(df)
    df = apply_interaction_maps(df, artifacts["interaction_maps"])
    df = apply_encoding_test(df, artifacts["te_maps"], artifacts["les"])

    print(f"  External test preprocessed: {df.shape}")
    return df



 MAIN

In [83]:
print("\n" + "="*60)
print("  DEMAND FORECASTING — LEAK-FREE PIPELINE  (v3)")
print("="*60)

    # ── 1. Load raw data ─────────────────────────────────────────────────────
df = load_and_parse(DATA_PATH )  #  load with the data path

    # ── 2. Split FIRST — before any demand-based feature ─────────────────────
train, test = split_train_test(df)

    # ── 3. Geohash decode ────────────────────────────────────────────────────
coords_map = fit_geohash_coords(train)
train = apply_geohash_coords(train, coords_map)
test  = apply_geohash_coords(test,  coords_map)

    # ── 4. Spatial clusters (fitted on train only) ───────────────────────────
km, cluster_map, dist_scaler = fit_spatial_clusters(train)
train = apply_spatial_clusters(train, km, cluster_map, dist_scaler)
test  = apply_spatial_clusters(test,  km, cluster_map, dist_scaler)

    # ── 5. Imputation (maps from train only) ─────────────────────────────────
imputation_maps = fit_imputation_maps(train)
print("\n[5] Applying imputation to train …")
train = apply_imputation(train, imputation_maps)
print("[5] Applying imputation to test  …")
test  = apply_imputation(test,  imputation_maps)

    # ── 6. Lag features (within each split, no cross-day leakage) ────────────
train = build_lag_features_train(train)
test  = build_lag_features_test(test, train)

    # ── 7. Cyclic time encoding ───────────────────────────────────────────────
print("\n[7] Cyclic time encoding …")
train = add_cyclic_and_time_flags(train)
test  = add_cyclic_and_time_flags(test)

    # ── 8. Domain flags ───────────────────────────────────────────────────────
print("\n[8] Domain flag features …")
train = add_domain_flags(train)
test  = add_domain_flags(test)

    # ── 9. Location × time interactions (train only) ─────────────────────────
interaction_maps = fit_interaction_maps(train)
train = apply_interaction_maps(train, interaction_maps)
test  = apply_interaction_maps(test,  interaction_maps)

    # ── 10. Categorical encoding ─────────────────────────────────────────────
train, te_maps, les = apply_encoding_train(train)
test  = apply_encoding_test(test, te_maps, les)

    # ── 11. Feature matrices ─────────────────────────────────────────────────
print("\n[11] Assembling feature matrices …")
X_train, y_train = get_feature_matrix(train)
X_test,  y_test  = get_feature_matrix(test)
print(f"    X_train: {X_train.shape}  |  X_test: {X_test.shape}")
print(f"    Features ({len(X_train.columns)}): {X_train.columns.tolist()}")

    # ── 12. Train ─────────────────────────────────────────────────────────────
model = train_lightgbm(X_train, y_train, X_test, y_test)

    # ── 13. Evaluate ──────────────────────────────────────────────────────────
metrics = evaluate(model, X_test, y_test, label="Test set (Day 49)", df_val=test)
print_feature_importance(model, X_train, top_n=20)

    # ── Save all artifacts ───────────────────────────────────────────────────
artifacts = {
        "coords_map":       coords_map,
        "km":               km,
        "cluster_map":      cluster_map,
        "dist_scaler":      dist_scaler,      # FIX #6
        "imputation_maps":  imputation_maps,
        "interaction_maps": interaction_maps,
        "te_maps":          te_maps,
        "les":              les,
        "feature_cols":     X_train.columns.tolist(),
    }
os.makedirs("artifacts", exist_ok=True)
with open("artifacts/pipeline_artifacts.pkl", "wb") as f:
  pickle.dump(artifacts, f)
model.save_model("artifacts/lgbm_model.txt")
print("\n✅  Model and artifacts saved to ./artifacts/")

    # ── Optional: preprocess a separate leaderboard test file ────────────────
if TEST_CSV_PATH and os.path.exists(TEST_CSV_PATH):
    df_ext = preprocess_external_test(TEST_CSV_PATH, train, artifacts)
    X_sub, _ = get_feature_matrix(df_ext)
    preds = np.clip(
          model.predict(X_sub, num_iteration=model.best_iteration), 0.0, 1.0
      )
    sub = pd.DataFrame({
        "Index":  df_ext.get("Index", range(len(preds))),
        "demand": preds,
    })
    sub.to_csv("submission.csv", index=False)
    print(f"✅  Submission saved: submission.csv  ({len(sub):,} rows)")

print("\n" + "="*60)
print("  PIPELINE COMPLETE")
print(f"  R²   : {metrics['r2']:.5f}")
print(f"  RMSE : {metrics['rmse']:.5f}")
print(f"  MAE  : {metrics['mae']:.5f}")
print("="*60 + "\n")

print(model, artifacts, metrics)





  DEMAND FORECASTING — LEAK-FREE PIPELINE  (v3)

[1] Loading '/content/drive/MyDrive/FLIPKART_GRIDLOCK/train.csv' …
    Raw shape: (77299, 11)

[2] Train/test split
    Train (day 48): 69,427 rows  (89.8%)
    Test  (day 49):  7,872 rows  (10.2%)
    ⚠  8 test geohashes not seen in train

[3] Decoding geohashes to lat/lon …
    Decoded 1,241 unique geohashes

[4] Fitting spatial KMeans (k=50) on train …
    50 spatial clusters created

[5] Fitting imputation maps on train …
    Train nulls before imputation: {'Temperature': 2241, 'RoadType': 539, 'Weather': 716}

[5] Applying imputation to train …
    Nulls after imputation: {'Temperature': 0, 'RoadType': 0, 'Weather': 0}
[5] Applying imputation to test  …
    Nulls after imputation: {'Temperature': 0, 'RoadType': 0, 'Weather': 0}

[6] Building lag features for train …
    Remaining NaNs in roll_std columns: 0  (expected 0)
    Total NaNs in all lag/rolling columns: 0  (expected 0)

[6-TEST] Building lag features for test (using train

EXP1    